# AI ANALYST LAB

![](../_img/Ghost_TheSyntheticBanner.png)

### A Hands-on Course on AI for Data Analysts
## Session 05: Binomial logistic regression for classification

Feedback should be sent to [goran.milovanovic@datakolektiv.com](mailto:goran.milovanovic@datakolektiv.com).

This notebook accompanies the **AI Analyst LAB** course. Welcome to Session 05 — the week we leave continuous outcomes behind and learn to predict a **yes/no** answer: *will this customer leave, or stay?*

***
### What we will do today

In the first four sessions we built the analyst's foundation. In **Session 01** we learned descriptive statistics, probability distributions (the **Normal**, the **Binomial**, the **Poisson**), and the sampling distribution of the mean. In **Session 02** we added conditional probability, expected value, and the bootstrap. In **Session 03** we built the hypothesis-testing framework — the chi-square and t-tests, and the meaning of a *p*-value. In **Session 04** we crossed into *relationships between numbers*: covariance, correlation, and **linear regression**, where we fit a straight line, read a coefficient's **t-test**, and measured how much variance the line explained with **$R^2$**.

At the very end of Session 04 we made you a promise:

> *"See you in Session 05, where we leave continuous outcomes behind and move into classification — predicting a yes/no outcome from a set of predictors. The $R^2$ and standard-error machinery from this week will reappear in a slightly different guise (deviance and log-odds), but the conceptual primitive — predict, take residuals, see what is left — carries straight through."*

This is that session. We are going to keep our promise **literally**: almost every new idea this week is the twin of something you already met in Session 04 (linear regression) or Session 01 (the Binomial distribution). When we hit a genuinely new and intimidating word — *log-odds*, *maximum likelihood*, *deviance* — we will walk back to the Session-04 idea it mirrors and build the new one on top of the old one. You will not need any calculus, and you will not need to have heard of *maximum likelihood* before today.

The business setting is a **mobile telecom company** losing customers to its competitors. The sections, in order:

| Section | What happens |
|---|---|
| 5.1 | The business case — the churn brief, and the "we can only call 10%" constraint |
| 5.2 | Meet your Session 05 Tutor (Claude Project) |
| 5.3 | Setup — imports, loading the churn data, cleaning the column names |
| 5.4 | Why a straight line **fails** for a yes/no outcome (a Session-04 callback) |
| 5.5 | **Probability, odds, and log-odds** — the three ways to say the same thing |
| 5.6 | **The logistic (sigmoid) function** — the bend that keeps predictions inside $[0,1]$ |
| 5.7 | **The logistic regression model** — and *why it is called "binomial"* (a Session-01 callback) |
| 5.8 | **Maximum likelihood** — the twin of least squares, with no calculus |
| 5.9 | Fit the model with **statsmodels** — the interpretable summary table |
| 5.10 | Reading the coefficients — signs and significance (the **z-test** is the twin of Session 04's t-test) |
| 5.11 | **Odds ratios** — turning each coefficient into a plain-English sentence |
| 5.12 | **Deviance** — *the new $SS_{res}$* (null deviance, residual deviance, pseudo-$R^2$) |
| 5.13 | A churn **probability for every customer** — and a look at `scikit-learn`'s `predict_proba` |
| 5.14 | From probability to **decision** — the threshold |
| 5.15 | **The confusion matrix** — the four ways a yes/no prediction can land |
| 5.16 | **Accuracy's trap**, and the metrics that escape it — sensitivity, specificity, precision, recall |
| 5.17 | **Moving the threshold** — the trade-off table |
| 5.18 | **The ROC curve and AUC** — grading the model at every threshold at once |
| 5.19 | The **"call only 10%"** policy — choosing a threshold to fit the budget |
| 5.20 | Our API calls — Anthropic explains the numbers and emits a structured **action policy** |
| 5.21 | The churn-triage memo — fully worked |
| 5.22 | References — what to study to deepen this session |

A few rules for using this notebook, same as the previous four weeks:

- **Run the cells in order.** Each section builds on the one before.
- **Read the explanations, do not just run the cells.** The intuition is the point; the formulas are the receipts.
- **Every line of code has a comment above it** in beginner language.
- **Use your Session 05 Tutor** (the Claude Project at `_tutors/session05_tutor.xml`) when something feels confusing.
- **Compute first in Python, then use the model to interpret.** Same rule as Sessions 01–04 — the model never invents numbers.

***
## 5.1 The business case

You have changed teams again. This week you are embedded with the **customer-retention team at a mobile telecom company**. Every month, some customers stop using the service and leave for a competitor. In the industry this is called **churn**, and it is expensive: winning a brand-new customer costs far more than keeping an existing one, so every customer who walks out of the door is a hole the sales team has to dig the company out of.

Your manager — the head of retention — drops a file on your desk on a Monday morning and says:

> *"We have last year's records for 3,150 customers, and for each one we know whether they ended up churning. I want to get ahead of it this year. Here is my problem: my team is small. We can phone, at most, about **10% of our customers** in a month with a retention offer — a discount, a better plan, a courtesy call. If I pick those customers at random, I waste almost all of my calls on people who were never going to leave. I need you to **score every customer by how likely they are to churn**, so I can spend my limited calls on the ones who are genuinely at risk. And I do not just want a yes/no flag — I want a **probability** I can rank people by, and I want you to be honest with me about how good the model actually is. Accuracy alone has burned me before."*

This is the canonical **classification** brief, and it is one of the most common analyst jobs in the world. *"Which customers will churn?"* *"Which transactions are fraud?"* *"Which leads will convert?"* *"Which patients are high-risk?"* — every one of them is the same shape: a **yes/no outcome** we want to predict from a handful of measurements, and a **limited budget** that forces us to act only on the most likely cases.

By the end of this session you will have:

1. **Loaded and cleaned** the Iranian Churn dataset (3,150 real telecom customers) with the same data-quality discipline we built in Sessions 01–04.
2. **Understood why linear regression cannot be used directly** for a yes/no outcome, and what to replace it with.
3. **Built the logistic regression model from the ground up** — probability, odds, log-odds, and the sigmoid bend — each one tied back to Session 04's straight line.
4. **Seen why the method is called "binomial"** — every customer is a single **Bernoulli trial**, exactly the building block of the Binomial distribution from Session 01.
5. **Understood how the model is fit** — *maximum likelihood*, taught as the twin of Session 04's least squares, with no calculus.
6. **Fit the model with `statsmodels`**, read every line of its summary, and judged each coefficient's significance with a **z-test** (the twin of Session 04's t-test).
7. **Turned coefficients into plain English** with **odds ratios**.
8. **Measured the model's fit with deviance** — null deviance, residual deviance, and a pseudo-$R^2$ — each one the direct analogue of Session 04's $SS_{tot}$, $SS_{res}$, and $R^2$.
9. **Produced a churn probability for every customer**, and seen `scikit-learn`'s `predict_proba` give the same answer.
10. **Turned probabilities into decisions** with a threshold, and graded those decisions with a **confusion matrix**, **sensitivity**, **specificity**, **precision**, and **recall** — escaping the trap of judging a model by accuracy alone.
11. **Drawn the ROC curve and computed AUC** to grade the model at every threshold at once.
12. **Chosen a threshold to fit the "call only 10%" budget**, and quantified how much better than random calling the model lets us do.
13. **Used Anthropic Claude** to explain the confusion matrix to a non-technical stakeholder, and to emit a structured **action policy** (risk tiers, thresholds, messages) via tool use — on numbers Python computed.

One thread weaves through the whole session: **a probability is not a decision.** The model gives us a number between 0 and 1 for every customer; turning that number into "call them / do not call them" is a separate, human choice with a cost on each side. Most of the second half of this notebook is about making that choice honestly. Let us get started.

***
## 5.2 Meet your Session 05 Tutor

Just like every previous week, this session comes with its own **Claude Project tutor** — a patient teaching assistant that knows exactly what we cover in Session 05 and will not race ahead to material you have not seen yet.

The tutor definition lives at `_tutors/session05_tutor.xml`. If you have not set up a tutor before, open `_tutors/TutorProjectCreation.md` for the step-by-step walkthrough; it takes about five minutes and you only do it once per session.

**What this tutor is for.** Ask it to re-explain anything in this notebook in different words: *"what is the difference between odds and probability?"*, *"why can't I just use accuracy?"*, *"explain deviance to me like I'm five, using the Session-04 analogy"*. It is scoped to **this session's statistics** — logistic regression, odds and log-odds, the confusion matrix, ROC/AUC, thresholds.

**What it will redirect.** If you ask it a deep `pandas` / `numpy` / `matplotlib` question, it will point you to the cross-session **Python stack tutor** (`_tutors/python_stack_tutor.xml`). If you ask about a statistics topic from an earlier week (the Binomial distribution, hypothesis testing, regression), it will point you back to that week's tutor. And like every tool in this course, it uses **Anthropic Claude** and nothing else.

> **A note on honesty.** Your tutor is a language model. It is excellent at re-explaining concepts, but — exactly like the API calls later in this notebook — it should never be the source of a *number*. Numbers come from Python, run on the real data. The tutor (and Claude) help you **understand and communicate** those numbers. Keep that division of labour in your head all week; it is the single most important professional habit in this course.

***
## 5.3 Setup — imports and loading the churn data

As always, we begin by importing the tools we will use and loading the data. This week's stack is the familiar Session-04 set — `pandas`, `numpy`, `matplotlib`, `seaborn` — plus two modelling libraries:

- **`statsmodels`** — the same library we used for the regression summary table in Session 04. It will fit our logistic regression and hand us a beautifully labelled table of coefficients, standard errors, and significance tests. This is our tool for **understanding** the model.
- **`scikit-learn`** (imported as parts of `sklearn`) — the most widely used machine-learning library in Python. We will use its ready-made tools for **evaluating** a classifier: the confusion matrix, the ROC curve, and the AUC score. We will also borrow its `LogisticRegression` to show it produces the same probabilities `statsmodels` does.

Run the cell below to bring everything into memory.

In [ ]:
# Import pandas under the alias pd; pandas gives us the DataFrame (a table of data).
import pandas as pd
# Import numpy under the alias np; numpy gives us fast arrays and math functions like exp and log.
import numpy as np
# Import matplotlib's pyplot under the alias plt; this is our basic plotting toolkit.
import matplotlib.pyplot as plt
# Import seaborn under the alias sns; seaborn makes prettier statistical charts on top of matplotlib.
import seaborn as sns

# Import statsmodels' high-level API as sm; this is the library that gives us the labelled summary table.
import statsmodels.api as sm
# Also import the top-level statsmodels package so we can read its version string.
import statsmodels

# From scikit-learn's metrics module, import the four evaluation tools we will need later.
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

# Tell Jupyter to draw charts directly underneath the cell that creates them.
%matplotlib inline
# Set a clean default chart style with a light grid, so every plot in this notebook looks consistent.
sns.set_theme(style="whitegrid")

# Print a short confirmation with the versions, so we know the libraries loaded correctly.
print("Libraries loaded.")
print("pandas", pd.__version__, "| numpy", np.__version__, "| statsmodels", statsmodels.__version__)

### Load the churn data

The dataset is the **Iranian Churn dataset** from the UCI Machine Learning Repository (the full citation and licence are in §5.22). It records **3,150 customers** of an Iranian mobile telecom company over a twelve-month window. For each customer we have a handful of behavioural measurements — how often they call, how many texts they send, how long they have been a subscriber, whether they have complained — and, crucially, a final column **`Churn`** that is **`1` if the customer left** and **`0` if they stayed**. That `Churn` column is our **outcome** — exactly the role the wine *quality* score played in Session 04, except now it is a yes/no flag instead of a number on a scale.

The raw file has one small annoyance: several column names contain **double spaces** (for example, `"Call  Failure"` with two spaces between the words). Double spaces are easy to mistype and awkward to refer to in code, so the very first thing we do after loading is **clean the column names** — strip surrounding spaces, collapse any run of spaces to a single space, and replace spaces with underscores so we get tidy names like `Call_Failure`.

In [ ]:
# Build the path to the CSV. The notebook lives in AI_AnalystLAB/Session05/, so ONE ".." steps up
# to the repo root, and from there we go into _data/iranian_churn/.
csv_path = "../_data/iranian_churn/Customer Churn.csv"

# Read the comma-separated file into a DataFrame called churn.
churn = pd.read_csv(csv_path)

# Define a small helper that tidies one column name:
def tidy(name):
    # strip() removes spaces at the very start and end of the name.
    name = name.strip()
    # " ".join(name.split()) collapses any run of whitespace down to a single space.
    name = " ".join(name.split())
    # replace(" ", "_") swaps the remaining single spaces for underscores.
    name = name.replace(" ", "_")
    # hand back the cleaned-up name.
    return name

# Apply the helper to every column name in one sweep, and overwrite the old names.
churn.columns = [tidy(c) for c in churn.columns]

# Show the cleaned column names so we can confirm the double-spaces are gone.
print(list(churn.columns))

### Four checks before trusting the data

Exactly as in Sessions 01–04, we never start analysing a fresh dataset without four quick sanity checks: **how big is it, what does a row look like, what type is each column, and is anything missing?** This discipline catches the silent disasters — a column read as text instead of a number, a file that only half-loaded — before they poison everything downstream.

In [ ]:
# Check 1 — shape. The .shape attribute returns a (rows, columns) pair.
churn.shape

You should see `(3150, 14)` — **3,150 customers** (rows) and **14 columns** (thirteen predictors plus the `Churn` outcome).

In [ ]:
# Check 2 — head. .head() shows the first five rows so we can eyeball a real record.
churn.head()

In [ ]:
# Check 3 — dtypes. Confirm every column is numeric (int or float), not text.
churn.dtypes

In [ ]:
# Check 4 — any missing values? .isna() flags blanks; .sum().sum() adds them all up to one number.
churn.isna().sum().sum()

You should see **0 missing values** across the entire table — this dataset is complete and clean, which is convenient and not always the case in the wild.

### How many customers actually churned?

Before any modelling, we need one number burned into our memory: **what fraction of customers churned?** This is called the **base rate** (or **prevalence**) of the outcome. It is the single most important context number for everything that follows, because it tells us how *rare* the thing we are predicting actually is — and, as we will see in §5.16, it sets the trap that a naive accuracy score falls straight into.

Because `Churn` is coded as `1` (left) and `0` (stayed), a wonderful shortcut applies: **the mean of a column of 0s and 1s is exactly the proportion of 1s.** If 495 out of 3,150 values are `1`, the mean is $495/3150$. So `churn["Churn"].mean()` hands us the base rate directly — a small callback to Session 01, where we first used the mean of a 0/1 indicator as a probability estimate.

In [ ]:
# Count how many customers fall in each Churn category: 0 = stayed, 1 = left.
counts = churn["Churn"].value_counts()
# Pull out the two counts by their labels so we can print them in plain language.
n_stay = int(counts[0])
n_churn = int(counts[1])
# The total number of customers is just the number of rows.
n_total = len(churn)
# The base rate is the mean of the 0/1 column — i.e. the proportion of 1s (churners).
base_rate = churn["Churn"].mean()

# Print all four numbers in a sentence a human can read.
print(f"Total customers : {n_total}")
print(f"Stayed (Churn=0): {n_stay}")
print(f"Left   (Churn=1): {n_churn}")
print(f"Base churn rate : {base_rate:.4f}  ({base_rate*100:.2f}%)")

**What we see.** Of the **3,150** customers, **495 churned** and **2,655 stayed** — a base churn rate of about **15.7%**. So churn is the *minority* outcome: roughly one customer in six. Hold on to that "one in six" — in §5.16 it will explain why a model that looks 84% accurate can be completely useless.

***
## 5.4 Why a straight line fails for a yes/no outcome

Let us start exactly where Session 04 left off, and watch it break — on purpose.

In Session 04 we fit a **straight line**:

$$ \hat{y} \;=\; \beta_0 + \beta_1 x . $$

Read aloud: *"y-hat equals beta-zero plus beta-one times x."* Here $\hat{y}$ (*"y-hat"*) is the model's prediction, $x$ is a predictor, $\beta_0$ (*"beta-zero"*) is the intercept — the prediction when $x = 0$ — and $\beta_1$ (*"beta-one"*) is the slope — how much the prediction moves when $x$ goes up by one unit. That worked beautifully when $y$ was a *quality score* that could be any number.

Now our outcome is **`Churn`**, which can only be **0 or 1**. What we actually want to predict is the **probability of churn**, a number that must live between 0 and 1. So the natural temptation is: just fit the straight line with `Churn` as $y$, and read $\hat{y}$ as a probability. Let us try it and see what goes wrong.

In [ ]:
# Pull out one intuitive predictor: how many distinct people the customer called.
# Low engagement (few contacts) plausibly relates to leaving, so it makes a good illustration.
x_demo = churn["Frequency_of_use"].values
# The outcome stays as the 0/1 Churn flag.
y_demo = churn["Churn"].values

# Fit an ordinary straight line (the Session-04 least-squares fit) through the 0/1 points.
# np.polyfit with degree 1 returns [slope, intercept].
slope, intercept = np.polyfit(x_demo, y_demo, 1)

# Build a smooth row of x-values spanning the data range, to draw the fitted line.
x_line = np.linspace(x_demo.min(), x_demo.max(), 200)
# Compute the line's predicted "probability" at each of those x-values.
y_line = intercept + slope * x_line

# Set up the figure.
plt.figure(figsize=(8, 5))
# Scatter the raw 0/1 outcomes against the predictor, with a little transparency since points overlap.
plt.scatter(x_demo, y_demo, alpha=0.15, s=15, label="actual Churn (0 or 1)")
# Draw the straight-line fit on top in red.
plt.plot(x_line, y_line, color="crimson", linewidth=2, label="straight-line fit")
# Draw faint reference lines at probability 0 and 1 — the only legal range for a probability.
plt.axhline(0.0, color="gray", linestyle=":", linewidth=1)
plt.axhline(1.0, color="gray", linestyle=":", linewidth=1)
# Label the axes and add a title.
plt.xlabel("Frequency_of_use (number of calls)")
plt.ylabel("Churn / predicted probability")
plt.title("A straight line through a 0/1 outcome leaves the [0, 1] box")
# Show the legend so the reader can tell the two elements apart.
plt.legend()
# Render the chart.
plt.show()

**What we see — and why it disqualifies the straight line.** Two problems, both fatal:

1. **The line leaves the box.** A probability must stay between 0 and 1, but a straight line goes on forever in both directions. For some customers the line predicts a "probability" *below 0* or *above 1*, which is meaningless. A probability of $-0.3$ or $1.4$ is not a probability at all.
2. **The shape is wrong.** When a customer is already almost certain to stay, sending their usage even higher should barely move the needle — they are already near probability 0. A straight line cannot do this; it changes by the same amount everywhere. Real probabilities *flatten out* as they approach 0 and 1.

So we need a model that (a) is built from the familiar linear piece $\beta_0 + \beta_1 x$, because that part worked, but (b) **bends** the output so it can never escape $[0, 1]$, and flattens gracefully at the two ends. The rest of §§5.5–5.7 builds exactly that bend. The trick will be to **not** put the straight line on the probability directly — instead we put it on a *transformed* version of the probability that genuinely can run from $-\infty$ to $+\infty$. That transformed version is the **log-odds**, and to understand it we first need **odds**.

***
## 5.5 Probability, odds, and log-odds — three ways to say the same thing

Here is the central idea of the whole session, and it is gentler than its vocabulary suggests. There are **three different scales** for expressing "how likely is churn?", and they are all just re-encodings of one another. We move between them constantly, so let us meet all three with a tiny, concrete example before any formula.

Imagine a single customer whose true probability of churning is **$p = 0.8$** (eight chances in ten of leaving). Three ways to say that:

- **Probability**: $p = 0.8$. *"Eighty percent chance of leaving."* Lives between 0 and 1.
- **Odds**: $0.8 / 0.2 = 4$. *"Four times more likely to leave than to stay"* — four-to-one *on* leaving. Lives between 0 and $+\infty$.
- **Log-odds**: $\ln(4) \approx 1.386$. The natural logarithm of the odds. Lives between $-\infty$ and $+\infty$.

That last scale — **log-odds** — is the prize. It is the only one of the three that can run from minus infinity to plus infinity, which means it is the only one a straight line can safely be glued to. Let us define each precisely.

### Odds

The **odds** of an event are the probability it happens divided by the probability it does not:

$$ \text{odds} \;=\; \frac{p}{1 - p}. $$

Reading every symbol: $p$ is the probability the event happens (here, churn). $1 - p$ is the probability it does **not** happen (the customer stays). Their ratio is the odds. A few anchor values worth memorising:

- $p = 0.5 \Rightarrow \text{odds} = 0.5/0.5 = 1$ — *"even odds", one-to-one.*
- $p = 0.8 \Rightarrow \text{odds} = 0.8/0.2 = 4$ — *"four-to-one on".*
- $p = 0.2 \Rightarrow \text{odds} = 0.2/0.8 = 0.25$ — *"one-to-four", i.e. four-to-one against.*

Notice the odds are never negative (both $p$ and $1-p$ are between 0 and 1), so odds run from $0$ up to $+\infty$. Better than probability — no upper ceiling — but still floored at zero. We need to remove that floor too.

### Log-odds (the logit)

Take the natural logarithm of the odds and you get the **log-odds**, also called the **logit**:

$$ \text{logit}(p) \;=\; \ln\!\left( \frac{p}{1 - p} \right). $$

Reading the symbols: $\ln$ (*"natural log"*) is the logarithm to base $e \approx 2.718$ — the same function `np.log` computes. The argument is the odds we just defined. Why the logarithm? Because the logarithm of a number between $0$ and $+\infty$ runs the **full** number line from $-\infty$ to $+\infty$:

- odds near $0$ (very unlikely) $\Rightarrow$ log-odds near $-\infty$,
- odds $= 1$ (even chance, $p = 0.5$) $\Rightarrow$ log-odds $= \ln(1) = 0$,
- odds very large (almost certain) $\Rightarrow$ log-odds near $+\infty$.

So **the log-odds is the scale with no ceiling and no floor** — exactly the unbounded canvas a straight line needs. This is the quiet hinge the entire method swings on: we will let the linear piece $\beta_0 + \beta_1 x$ predict the **log-odds**, not the probability, and then convert back to a probability at the end. Let us confirm the three scales line up with a direct computation.

In [ ]:
# Define a tiny helper that prints all three scales for a given probability p.
def show_scales(p):
    # odds = p / (1 - p): how many times more likely the event is than its absence.
    odds = p / (1 - p)
    # log-odds = natural log of the odds; np.log is the natural logarithm.
    logodds = np.log(odds)
    # Print the three numbers side by side, rounded for readability.
    print(f"p = {p:>4}   odds = {odds:>6.3f}   log-odds = {logodds:>7.3f}")

# Walk through five anchor probabilities from unlikely to very likely.
for p in [0.1, 0.2, 0.5, 0.8, 0.9]:
    show_scales(p)

**What we see.** Read the middle row: $p = 0.5$ gives odds $= 1.0$ and log-odds $= 0.0$ exactly — the natural centre. Above $0.5$ the log-odds are positive; below it they are negative; and as $p$ creeps toward 0 or 1 the log-odds shoot off toward $-\infty$ or $+\infty$. Notice also the pleasing symmetry: $p=0.2$ gives log-odds $-1.386$ and $p=0.8$ gives $+1.386$ — the same distance from zero, opposite signs. That symmetry is the log-odds scale being fair to "leaving" and "staying". This table *is* the bridge between the human-friendly probability scale and the model-friendly log-odds scale. Now we build the bend that carries us back the other way.

***
## 5.6 The logistic (sigmoid) function — the bend that keeps us inside $[0, 1]$

In §5.5 we travelled from probability to log-odds. Now we need the **return journey**: given a log-odds value (any number on the whole line), what probability does it correspond to? Inverting the logit formula gives the famous **logistic function**, also called the **sigmoid** (from the Greek for "S-shaped"):

$$ p \;=\; \sigma(z) \;=\; \frac{1}{1 + e^{-z}}. $$

Reading every symbol: $\sigma$ (the Greek letter *"sigma"*) is the name of the function. $z$ is the input — a log-odds value, which can be any number from $-\infty$ to $+\infty$. $e \approx 2.718$ is Euler's number, the base of the natural logarithm. $e^{-z}$ (*"e to the minus z"*) is that constant raised to the power $-z$ — exactly what `np.exp(-z)` computes. The output $p$ is a probability, and — this is the whole point — **no matter what $z$ you put in, the output is always strictly between 0 and 1.** The bend is built into the algebra.

Three anchor values, which you can check against the §5.5 table read backwards:

- $z = 0 \Rightarrow \sigma(0) = 1/(1 + 1) = 0.5$. *Log-odds of zero means even chance — agrees with §5.5.*
- $z = +2 \Rightarrow \sigma(2) \approx 0.881$. *A solidly positive log-odds means likely.*
- $z = -2 \Rightarrow \sigma(-2) \approx 0.119$. *A solidly negative log-odds means unlikely.*

The sigmoid is precisely the inverse of the logit: `logit` carries a probability **up** to the unbounded log-odds scale; `sigmoid` carries a log-odds value back **down** to a probability. Let us draw it.

In [ ]:
# Define the sigmoid function exactly as written in the formula above.
def sigmoid(z):
    # 1 / (1 + e^{-z}); np.exp is the exponential function e^(...).
    return 1.0 / (1.0 + np.exp(-z))

# Build a row of log-odds values from -6 to +6 to feed the curve.
z_grid = np.linspace(-6, 6, 300)
# Apply the sigmoid to every value to get the matching probabilities.
p_grid = sigmoid(z_grid)

# Set up the figure.
plt.figure(figsize=(8, 5))
# Plot the S-shaped curve: probability (y) against log-odds (x).
plt.plot(z_grid, p_grid, color="steelblue", linewidth=2.5)
# Mark the centre point (z=0, p=0.5) — the natural tipping point.
plt.scatter([0], [0.5], color="crimson", zorder=5)
# Annotate that centre point.
plt.annotate("z = 0 -> p = 0.5", xy=(0, 0.5), xytext=(1, 0.35),
             arrowprops=dict(arrowstyle="->", color="crimson"))
# Draw faint reference lines at the legal probability limits 0 and 1.
plt.axhline(0.0, color="gray", linestyle=":", linewidth=1)
plt.axhline(1.0, color="gray", linestyle=":", linewidth=1)
# Label the axes and title the chart.
plt.xlabel("z  =  log-odds  (the linear predictor)")
plt.ylabel("p  =  probability of churn")
plt.title("The logistic (sigmoid) function: any z maps to a probability in (0, 1)")
# Render the chart.
plt.show()

# Print the three anchor values so the numbers are on the page, not just in the picture.
for z in [-2, 0, 2]:
    print(f"sigmoid({z:+d}) = {sigmoid(z):.4f}")

**What we see.** The curve never touches 0 or 1 — it only *approaches* them, flattening at both ends. That flattening is the second thing the straight line could not do (§5.4): once a customer is nearly certain to stay, pushing $z$ even more negative barely changes $p$, because the curve is already hugging the floor. The steepest part — where the model is most "undecided" and a small change in $z$ moves the probability the most — is right around $z = 0$, $p = 0.5$. This single S-shaped function is the engine of logistic regression.

***
## 5.7 The logistic regression model — and why it is called "binomial"

We now have all the parts. Let us assemble the model and then explain its full name, *binomial logistic regression*, one word at a time.

### The model

Recall the linear piece from Session 04, $\beta_0 + \beta_1 x$. With several predictors $x_1, x_2, \ldots, x_k$ it generalises to the **linear predictor**:

$$ z \;=\; \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_k x_k. $$

This is *identical* to Session 04's multiple regression right-hand side — same intercept $\beta_0$, same slopes $\beta_1 \ldots \beta_k$, same predictors. Reading the symbols: $z$ is the linear predictor; $\beta_0$ is the intercept; each $\beta_j$ is the slope for predictor $x_j$; and $k$ is how many predictors we use.

In Session 04 this expression *was* the prediction. In logistic regression we make one change, the change that fixes everything in §5.4: **we declare that $z$ is the log-odds, not the probability.** Then we pass it through the sigmoid to get the probability:

$$ \ln\!\left( \frac{p}{1-p} \right) = z = \beta_0 + \beta_1 x_1 + \cdots + \beta_k x_k, \qquad\text{equivalently}\qquad p = \sigma(z) = \frac{1}{1 + e^{-z}}. $$

Read the left equation aloud: *"the log-odds of churn is a straight-line combination of the predictors."* That is the entire model. **The linearity you learned in Session 04 is still here — it just lives on the log-odds scale now, and the sigmoid translates it into a probability at the end.** Everything else this week is about (a) finding good values for the $\beta$s, and (b) judging how well they work.

### Why "binomial"? A Session-01 callback

The method's full name is **binomial logistic regression**, and the word *binomial* is not decoration — it points straight back to **Session 01**.

Remember the **Binomial distribution** from Session 01, §1.6? We built it from a simpler atom: the **Bernoulli trial** — a single yes/no experiment, like one coin flip, that comes up "success" with probability $p$ and "failure" with probability $1 - p$. The Binomial distribution then counted the successes across $n$ independent Bernoulli trials that all shared the *same* $p$.

A churn dataset is a stack of Bernoulli trials. **Each customer is one trial**: they either churn (success, $y = 1$) or stay (failure, $y = 0$). The probability mass function for a single customer is the Bernoulli one we met in Session 01:

$$ P(Y = y) \;=\; p^{\,y}\,(1 - p)^{\,1 - y}, \qquad y \in \{0, 1\}. $$

Reading the symbols: $Y$ (*"big why"*) is the random yes/no outcome for one customer; $y$ is its realised value, either 0 or 1; $p$ is that customer's churn probability. Check it: if $y = 1$ the formula returns $p^1 (1-p)^0 = p$; if $y = 0$ it returns $p^0 (1-p)^1 = 1 - p$. Exactly right.

Here is the **one twist** that turns Session 01's Binomial into a *regression*. In Session 01 every trial shared a single common $p$. In logistic regression **each customer gets their own $p_i$**, computed from *their* predictor values through the sigmoid:

$$ p_i \;=\; \sigma\!\left( \beta_0 + \beta_1 x_{i1} + \cdots + \beta_k x_{ik} \right). $$

The subscript $i$ indexes the customer; $x_{ij}$ is customer $i$'s value of predictor $j$. So *binomial logistic regression* literally means: **a pile of Bernoulli/Binomial trials (Session 01), where each trial's success probability is steered by a logistic regression on that individual's predictors.** The name is a precise description, and you already owned half of it four weeks ago. That is why we can say, honestly, that this week is less a new building than a new floor on the Session-01 and Session-04 foundations.

***
## 5.8 How the line is fit — maximum likelihood, the twin of least squares

We have a model with unknown coefficients $\beta_0, \beta_1, \ldots, \beta_k$. How do we choose their *values* from the data? In Session 04 we had a crisp answer for the straight line — **least squares** — and the good news is that logistic regression's answer is its **direct twin**. If you understood least squares, you already understand the *shape* of what follows; only the scorecard changes. No calculus is required to grasp the idea — we will not differentiate anything.

### Reminder: what least squares did (Session 04)

For the straight line, every data point had a **residual** — the vertical miss between the actual value and the line's prediction. We squared each residual (so misses above and below count equally and big misses count a lot), added them up into the **residual sum of squares** $SS_{res}$, and then chose the slope and intercept that made that sum **as small as possible**. In one sentence: *least squares picks the line that misses the data by the least.*

### The twist for a yes/no outcome

For a 0/1 outcome, "vertical miss" is clumsy — what is the miss when the truth is "churned" and the model says "probability 0.7"? Instead we score a fit by a more natural question:

> **Given these coefficients, how plausible is the exact pattern of 0s and 1s we actually observed?**

The model hands each customer a churn probability $p_i$. For a customer who **actually churned**, the model "spent" probability $p_i$ on that correct outcome — the bigger $p_i$, the more the model expected what really happened. For a customer who **actually stayed**, the model spent $1 - p_i$ on the correct outcome. The single number that scores the whole dataset is the **likelihood** — the probability the model assigns to the *entire observed sequence* of outcomes, which (because the customers are independent Bernoulli trials, §5.7) is the **product** of each customer's probability-of-what-actually-happened:

$$ \mathcal{L} \;=\; \prod_{i=1}^{n} \; p_i^{\,y_i}\,(1 - p_i)^{\,1 - y_i}. $$

Reading the symbols: $\mathcal{L}$ is the likelihood; $\prod$ means "multiply all of these together", the running-product cousin of the summation $\sum$; $i$ runs over all $n$ customers; $y_i$ is customer $i$'s actual 0/1 outcome; $p_i$ is the churn probability the model gave them. The exponents are the same Bernoulli trick from §5.7: when $y_i = 1$ the term is $p_i$, and when $y_i = 0$ the term is $1 - p_i$ — so **every term is just the probability the model put on the thing that actually happened.**

**Maximum likelihood estimation (MLE)** is then the exact mirror image of least squares:

| | Session 04: least squares | Session 05: maximum likelihood |
|---|---|---|
| Each point contributes… | a squared residual | a probability of its actual outcome |
| Combine them by… | **adding** ($SS_{res}$) | **multiplying** ($\mathcal{L}$) |
| Choose coefficients to… | **minimise** the total | **maximise** the total |
| In one phrase… | *"miss the data by the least"* | *"be least surprised by what actually happened"* |

That is the whole idea: **least squares finds the line that misses by the least; maximum likelihood finds the coefficients that are least surprised by the data.** Same spirit — make the model fit the observed data as well as it possibly can — different scorecard, because the outcome changed from a number to a yes/no.

### One practical wrinkle: we add logs instead of multiplying

Multiplying 3,150 numbers that are each below 1 produces a fantastically tiny result — far too small for a computer to hold without rounding it to zero. The standard fix is the one statisticians always reach for: **take the logarithm**, which turns a product into a sum. The **log-likelihood** is

$$ \ln \mathcal{L} \;=\; \sum_{i=1}^{n} \Big[\, y_i \ln(p_i) + (1 - y_i)\ln(1 - p_i) \,\Big]. $$

Because the logarithm only ever climbs as its input grows, the coefficients that **maximise the likelihood** are exactly the ones that **maximise the log-likelihood** — we lose nothing by working with logs, and we gain a sum we can actually compute. When `statsmodels` reports `Log-Likelihood` in the next section, *this* is the number it means, and "fitting the model" means searching for the $\beta$ values that push this sum as high as it will go. (The search itself is done by the computer; it is the numerical cousin of how Session 04 solved for the least-squares line, and we happily leave the machinery to the library.)

Let us make the idea concrete with a two-customer toy before we fit the real thing.

In [ ]:
# A tiny world with just TWO customers, to see likelihood in action.
# Customer A actually churned (y=1); customer B actually stayed (y=0).
y_toy = np.array([1, 0])

# Candidate model 1 — a GOOD guess: it thinks A is likely to churn (0.8) and B is likely to stay (so 0.1 churn).
p_good = np.array([0.8, 0.1])
# Candidate model 2 — a BAD guess: it has them almost backwards (A only 0.3, B a worrying 0.6).
p_bad  = np.array([0.3, 0.6])

# Define the likelihood of the observed outcomes under a set of probabilities.
def likelihood(y, p):
    # For each customer: p if they churned (y=1), else (1 - p) if they stayed (y=0).
    per_customer = np.where(y == 1, p, 1 - p)
    # The whole-dataset likelihood is the product of the per-customer probabilities.
    return per_customer.prod(), per_customer

# Compute likelihood under each candidate model.
L_good, terms_good = likelihood(y_toy, p_good)
L_bad,  terms_bad  = likelihood(y_toy, p_bad)

# Print the per-customer "probability of what actually happened" and the total for each model.
print("GOOD model: per-customer prob of actual outcome =", np.round(terms_good, 3),
      "  ->  likelihood =", round(L_good, 4))
print("BAD  model: per-customer prob of actual outcome =", np.round(terms_bad, 3),
      "  ->  likelihood =", round(L_bad, 4))
print()
print("The GOOD model has the higher likelihood, so MLE would prefer it.")

**What we see.** The good model assigns probability $0.8$ to A's churn and $0.9$ to B's staying, for a likelihood of $0.8 \times 0.9 = 0.72$. The bad model manages only $0.3 \times 0.4 = 0.12$. Maximum likelihood simply turns this comparison into a search: out of *all possible* coefficient values, find the ones whose implied probabilities make the observed 0/1 pattern as plausible as possible. The winner is our fitted model. Now let us let `statsmodels` run that search on the real 3,150 customers.

***
## 5.9 Fit the model with `statsmodels`

We now fit the model on **all 3,150 customers**, using `statsmodels` — the same library that gave us the labelled regression table in Session 04. The recipe is almost identical to that week:

1. choose the predictor columns and the outcome column,
2. add an intercept column of 1s (so the model can learn $\beta_0$) — `statsmodels` does not add it automatically, exactly as in Session 04,
3. hand both to `sm.Logit(...)` (Logit = the logistic model) and call `.fit()`.

We will use **eleven predictors**, chosen to be readable behavioural facts about a customer: whether they have complained, how long they have subscribed, their charge tier, their usage seconds and call/SMS frequency, how many distinct numbers they called, their account status, their customer-value score, their call-failure count, and their age. The point of this session is the *method*, not predictor-hunting, so we keep a generous, plain-language set and let the significance tests tell us which ones earn their place.

> **A note on what we are (and are not) doing.** We fit on the full dataset and we will evaluate on the full dataset, framing every result as *"how well does this model describe these 3,150 customers?"* That is the right question for an analyst characterising the data in front of them, and it keeps this session focused on understanding the model rather than on the separate machinery of forecasting brand-new customers.

In [ ]:
# The eleven predictor columns we will model churn on (all plain behavioural facts).
predictors = [
    "Complains",                 # 1 if the customer lodged a complaint, else 0
    "Subscription_Length",       # how many months they have been a subscriber
    "Charge_Amount",             # an ordinal charge tier (0 = lowest ... up)
    "Seconds_of_Use",            # total seconds of calls
    "Frequency_of_use",          # number of calls
    "Frequency_of_SMS",          # number of text messages
    "Distinct_Called_Numbers",   # how many different numbers they called
    "Status",                    # account status code (1 = active, 2 = non-active)
    "Customer_Value",            # a computed value-to-company score
    "Call_Failure",              # number of failed calls they experienced
    "Age",                       # customer age in years
]

# Build the predictor matrix X by selecting those columns, as floats for the solver.
X = churn[predictors].astype(float)
# The outcome y is the 0/1 Churn flag.
y = churn["Churn"].astype(int)

# Add an intercept column of 1s so the model can estimate beta_0 (statsmodels needs this explicitly).
X_const = sm.add_constant(X)

# Create the logistic-regression model object: outcome first, then the design matrix.
logit_model = sm.Logit(y, X_const)
# Fit it by maximum likelihood (the search from 5.8). disp=False hides the iteration log.
logit_res = logit_model.fit(disp=False)

# Print the full, labelled summary table.
print(logit_res.summary())

**What we see — the headline numbers.** The model converged (it found the maximum-likelihood coefficients). A few orientation points before we read individual rows:

- **No. Observations: 3150** — all our customers, as promised.
- **Log-Likelihood: −692.65** — the peak of the log-likelihood sum from §5.8. On its own it is just a number; in §5.12 we will turn it into something interpretable.
- **LL-Null: −1369.94** — the log-likelihood of a *baseline* model that uses no predictors at all (it just predicts the overall 15.7% churn rate for everyone). Our model's −692.65 is much higher (less negative), which means our predictors genuinely helped.
- **Pseudo R-squ.: 0.4944** — a logistic cousin of Session 04's $R^2$, which we unpack in §5.12.
- **LLR p-value: 7.6e-284** — an astronomically tiny number testing *"do the predictors collectively help at all?"* The answer is an emphatic yes.

Now the individual coefficients.

***
## 5.10 Reading the coefficients — signs and significance

Each row of the `coef` column is a $\beta$ on the **log-odds scale**. Their *signs* are immediately readable, even before we convert anything:

- A **positive** coefficient pushes the log-odds **up** → higher churn probability as that predictor grows.
- A **negative** coefficient pushes the log-odds **down** → lower churn probability as that predictor grows.

Reading our table that way tells a coherent story:

- **`Complains` = +4.05** — by far the largest effect. Customers who have complained are dramatically more likely to churn. (A complaint is a cry for help that often precedes the door.)
- **`Status` = +1.41** — non-active-status customers are much more likely to churn. Sensible: disengagement precedes departure.
- **`Call_Failure` = +0.13** — more dropped calls, more churn. A clean signal of a frustrating experience.
- **`Customer_Value` = +0.008** — a small positive nudge.
- The **protective** predictors, all *negative*: **`Frequency_of_use` = −0.055**, **`Frequency_of_SMS` = −0.047**, **`Charge_Amount` = −0.40**, **`Subscription_Length` = −0.030**. The more someone actually uses the service (calls, texts), the longer they have been around, and the higher their charge tier, the **less** likely they are to leave. Engagement and tenure are sticky.

### Significance — the z-test is the twin of Session 04's t-test

Look at the `z` and `P>|z|` columns. This is the *same idea* as the `t` and `P>|t|` columns you read in Session 04, with one word changed. In Session 04 we asked of each slope: *"how many standard errors does our estimate sit away from zero?"* — and called that ratio a **t-statistic**. Here we ask the identical question:

$$ z \;=\; \frac{\hat{\beta}}{\text{SE}(\hat{\beta})}. $$

Reading the symbols: $\hat{\beta}$ (*"beta-hat"*) is the estimated coefficient; $\text{SE}(\hat{\beta})$ is its **standard error** — the same Session-01/Session-04 idea of *"how much would this estimate wobble if we redid the study?"*; their ratio is the **z-statistic**. (It is called *z* rather than *t* only because of a technical difference in how the reference distribution is justified for maximum-likelihood estimates — the **interpretation is exactly the one you already learned**: a big absolute *z*, and the matching tiny $P>|z|$, means the coefficient is reliably different from zero.) This particular test on a single coefficient is known as the **Wald test**.

By that standard, **`Complains`, `Status`, `Call_Failure`, `Frequency_of_use`, `Frequency_of_SMS`, `Charge_Amount`, `Subscription_Length` and `Customer_Value` are all strongly significant** (p-values from essentially 0 up to 0.003). Three predictors are **not** significant at the usual 0.05 line — **`Seconds_of_Use`** (p ≈ 0.49), **`Distinct_Called_Numbers`** (p ≈ 0.24) and **`Age`** (p ≈ 0.29). Their effect is statistically indistinguishable from zero once the other predictors are in the model. That is a useful, honest finding — not every plausible-sounding variable carries its weight, and the same hypothesis-testing logic from Session 03 (*"is this distinguishable from no effect?"*) is doing the work.

***
## 5.11 Odds ratios — turning each coefficient into a plain-English sentence

A coefficient of $+4.05$ on the log-odds scale is correct but useless in a meeting. We need to say it in words a retention manager understands. The bridge is the **odds ratio**, and it comes from one move you already know: in §5.5 we saw that exponentiating undoes a logarithm.

Because each $\beta_j$ is the change in **log-odds** for a one-unit increase in predictor $x_j$, raising $e$ to that coefficient converts it into a multiplier on the **odds** themselves:

$$ \text{odds ratio for } x_j \;=\; e^{\beta_j}. $$

Reading it: $e^{\beta_j}$ is the factor by which the **odds of churn get multiplied** when $x_j$ goes up by one unit, holding the other predictors fixed (the same *"all else equal"* clause from Session 04's multiple regression). The interpretation key:

- odds ratio **$> 1$**: that predictor **raises** the odds of churn (a risk factor);
- odds ratio **$= 1$**: no effect;
- odds ratio **$< 1$**: that predictor **lowers** the odds of churn (a protective factor).

Let us compute them for every predictor.

In [ ]:
# Exponentiate each fitted coefficient to turn log-odds units into odds-multipliers.
odds_ratios = np.exp(logit_res.params)

# Assemble a tidy table: the coefficient, its odds ratio, and its p-value side by side.
or_table = pd.DataFrame({
    "coef (log-odds)": logit_res.params,   # the raw beta on the log-odds scale
    "odds_ratio": odds_ratios,             # e^beta: the multiplier on the odds
    "p_value": logit_res.pvalues,          # the Wald-test p-value from 5.10
})
# Round for readability and show the table.
print(or_table.round(4).to_string())

**What we see — read three of these aloud.**

- **`Complains`, odds ratio ≈ 57.6.** Having complained multiplies a customer's *odds* of churning by about **57**. This is enormous — complainers are in a different universe of risk. (Remember the raw cross-tab intuition: in this data about 83% of complainers churned versus about 10% of non-complainers.) Because `Complains` is a 0/1 variable, this odds ratio compares complainers directly to non-complainers.
- **`Status`, odds ratio ≈ 4.11.** Each step up the status code (active → non-active) multiplies the odds of churn by about **4**.
- **`Frequency_of_use`, odds ratio ≈ 0.947.** Below 1, so it is **protective**: each additional call multiplies the odds of churn by 0.947 — i.e. shaves about **5% off the odds**. One call barely matters; a hundred extra calls compounds into a large protective effect, because the multipliers stack.

For the continuous predictors a "one-unit" change can be tiny (one extra second of use, one extra month), so odds ratios very close to 1 are expected and still meaningful once you scale them up to a realistic change. The sign and the rough magnitude are what you carry into the room: **complaints and inactive status are the big red flags; genuine usage and tenure are the green ones.** This is the model's qualitative story — and notice we have not yet made a single prediction. That is next.

***
## 5.12 Deviance — *the new $SS_{res}$*

This is the section the whole "twin of Session 04" framing was built for. The word **deviance** sounds like new mathematics; it is not. It is the logistic-regression stand-in for a quantity you already understand completely — Session 04's **residual sum of squares** — and once you see them side by side, the mystery evaporates. We cannot explain deviance honestly *from scratch* to someone who has never met it; what we *can* do is hand you the Session-04 idea it copies, joint for joint.

### Reminder: how Session 04 measured fit

In Session 04 we measured a regression's fit with a variance-accounting decomposition:

- **$SS_{tot}$** (total sum of squares) — the total variation in the outcome around its mean. This is the "badness" you are stuck with if you ignore the predictors and just guess the average for everyone. It is the **baseline**.
- **$SS_{res}$** (residual sum of squares) — the variation *left over* after the line does its best. The **leftover badness**.
- **$R^2 = 1 - \dfrac{SS_{res}}{SS_{tot}}$** — the fraction of the baseline badness the model managed to remove. $R^2 = 1$ is perfect; $R^2 = 0$ means the predictors bought you nothing.

### The logistic copy, joint for joint

For a 0/1 outcome there are no squared residuals to add up, so we need a different "badness" number. We already have the perfect candidate from §5.8: the **log-likelihood** measures how *plausible* the data is under a model — and bigger (less negative) is better. To turn "plausibility (bigger is better)" into "badness (smaller is better)", we multiply by $-2$. That number has a name: the **deviance**.

$$ \text{deviance} \;=\; -2 \, \ln \mathcal{L}. $$

The minus sign flips "big is good" into "small is good"; the factor of 2 is a convention that makes deviance *differences* behave like the **chi-square** statistic from Session 03 (a callback we cash in below). With that one definition, the entire Session-04 table translates directly:

| Session 04 (linear regression) | Session 05 (logistic regression) | Meaning |
|---|---|---|
| $SS_{tot}$ — baseline badness (guess the mean) | **Null deviance** $= -2\,\ln\mathcal{L}_{\text{null}}$ | how badly the no-predictor model fits |
| $SS_{res}$ — leftover badness after the fit | **Residual deviance** $= -2\,\ln\mathcal{L}_{\text{model}}$ | how badly our model still fits |
| $R^2 = 1 - \dfrac{SS_{res}}{SS_{tot}}$ | **pseudo-$R^2$** $= 1 - \dfrac{\text{residual deviance}}{\text{null deviance}}$ | fraction of baseline badness removed |

Read the table across: **null deviance is the new $SS_{tot}$, residual deviance is the new $SS_{res}$, and McFadden's pseudo-$R^2$ is the new $R^2$ — defined by the *exact same* "one minus leftover-over-total" formula.** Let us compute all three from the quantities `statsmodels` already gave us and confirm they line up.

In [ ]:
# The model's log-likelihood (the peak from 5.8) — statsmodels stores it as .llf.
ll_model = logit_res.llf
# The null (no-predictor) model's log-likelihood — statsmodels stores it as .llnull.
ll_null = logit_res.llnull

# Deviance = -2 * log-likelihood. Residual deviance uses the model; null deviance uses the baseline.
residual_deviance = -2 * ll_model
null_deviance = -2 * ll_null

# McFadden's pseudo-R^2, written the SAME way as Session 04's R^2: 1 - leftover / total.
pseudo_r2 = 1 - residual_deviance / null_deviance

# Print the three numbers and the pseudo-R^2.
print(f"Null deviance     (the new SS_tot): {null_deviance:8.2f}")
print(f"Residual deviance (the new SS_res): {residual_deviance:8.2f}")
print(f"Deviance removed by our predictors: {null_deviance - residual_deviance:8.2f}")
print(f"Pseudo-R^2 = 1 - resid/null       : {pseudo_r2:8.4f}")

# Cross-check: statsmodels computes the same pseudo-R^2 directly as .prsquared.
print(f"statsmodels .prsquared (cross-check): {logit_res.prsquared:8.4f}")

**What we see.** The null deviance is about **2739.9** — the "badness" of guessing the flat 15.7% churn rate for everyone. Our eleven predictors pull the residual deviance down to about **1385.3**, removing roughly **1354.6** units of deviance. Expressed as a fraction, that is a pseudo-$R^2$ of about **0.494** — and it agrees to the decimal with `statsmodels`' own `.prsquared`, exactly as $R^2 = 1 - SS_{res}/SS_{tot}$ did in Session 04.

**How big is 0.494?** Here is the one place the analogy needs a footnote. McFadden's pseudo-$R^2$ does **not** read on the same generous scale as Session 04's $R^2$ — values around **0.2 to 0.4 already indicate an excellent logistic fit**, so 0.494 is genuinely strong. Do not compare a logistic pseudo-$R^2$ against the $R^2$ of a linear model and feel disappointed; they live on different rulers. Use it the way it is meant to be used: to compare *competing logistic models for the same outcome* (higher is better), and as a single honest summary of how much of the baseline uncertainty the predictors resolved.

**The Session-03 callback hiding in plain sight.** Remember the `LLR p-value` from the summary (§5.9)? That test compares the null deviance to the residual deviance: the *drop* of 1354.6 is, under the null hypothesis that none of the predictors matter, a draw from a **chi-square distribution** — the very distribution we built tests around in Session 03. A drop that large is so far into the tail that the p-value is about $10^{-284}$. So "do my predictors collectively help?" is answered by the same chi-square logic you already own; deviance is simply the quantity it is applied to.

***
## 5.13 A churn probability for every customer

So far we have *understood* the model. Now we put it to work. Feeding each customer's predictor values through the fitted model and the sigmoid (§5.6) produces their personal churn probability $p_i$. `statsmodels` does the whole chain for us with `.predict()`. This is the deliverable the retention manager actually asked for in §5.1: **a score for every customer that we can rank.**

In [ ]:
# Push every customer's predictors through the fitted model to get their churn probability.
# .predict() applies the linear predictor AND the sigmoid, returning a number in (0, 1) per customer.
churn_prob = logit_res.predict(X_const)

# Attach the probability back onto the DataFrame as a new column, for easy inspection.
churn["churn_prob"] = churn_prob

# Draw a histogram of the 3,150 predicted probabilities to see how the model spreads customers out.
plt.figure(figsize=(8, 5))
# 40 bins gives a detailed shape; the color is just for readability.
plt.hist(churn_prob, bins=40, color="slateblue", edgecolor="white")
# A reference line at the 15.7% base rate, for context.
plt.axvline(base_rate, color="crimson", linestyle="--", label=f"base rate = {base_rate:.3f}")
# Label axes and title.
plt.xlabel("predicted probability of churn")
plt.ylabel("number of customers")
plt.title("How the model scores all 3,150 customers")
plt.legend()
plt.show()

# Print a five-number summary of the probabilities so the spread is on the page too.
print(churn_prob.describe().round(4).to_string())

**What we see.** The model does **not** give everyone the same 15.7% — it spreads customers across the whole range. A large pile sits near zero (customers it is confident will stay) and a smaller group is pushed up toward 1 (the at-risk tail the retention team is hunting for). That separation is exactly what makes a probability *useful*: if the model gave everyone 0.157, ranking would be impossible and we would be back to calling at random. Let us eyeball the extremes to sanity-check the story.

In [ ]:
# Columns to display for a quick human sanity-check of the extremes.
peek_cols = ["Complains", "Status", "Frequency_of_use", "Call_Failure", "Churn", "churn_prob"]

# The five customers the model is MOST worried about (highest churn probability).
print("Top 5 highest-risk customers:")
print(churn.nlargest(5, "churn_prob")[peek_cols].to_string())
print()
# The five customers the model is LEAST worried about (lowest churn probability).
print("Bottom 5 lowest-risk customers:")
print(churn.nsmallest(5, "churn_prob")[peek_cols].to_string())

**What we see.** The highest-risk customers tend to be complainers and/or non-active, with low usage — and most of them did in fact churn. The lowest-risk customers are engaged, active, complaint-free — and stayed. The model's probabilities are tracking the human story sensibly, which is the qualitative check you should always do before trusting any score.

### The same probabilities from `scikit-learn`'s `predict_proba`

`statsmodels` gave us a model we can *read*. `scikit-learn` is the library you will more often meet for *prediction* in industry, and its classifiers expose a method called **`predict_proba`** — *"predict probabilities"* — that returns exactly the kind of per-customer probability we just computed. It is worth seeing that the two libraries, asked to fit the same logistic model, **agree**.

One small preparatory step: `scikit-learn`'s solver is happiest when the predictors are on a **common scale**, so we first pass the data through a `StandardScaler`, which simply re-expresses each predictor as *"how many spreads above or below its own average"* (subtract the mean, divide by the standard deviation — the same standardisation we used in Session 04 to build correlation). Scaling does not change *which* model is being fit, only how smoothly the solver gets there; the probabilities that come out are on the same 0-to-1 churn scale as before. We chain the scaler and the classifier together with `make_pipeline` so they always run as a unit.

In [ ]:
# Import scikit-learn's logistic-regression classifier.
from sklearn.linear_model import LogisticRegression
# Import the standardiser that puts every predictor on a common scale.
from sklearn.preprocessing import StandardScaler
# Import make_pipeline, which glues preprocessing and model into one object.
from sklearn.pipeline import make_pipeline

# Build a pipeline: first standardise the predictors, then fit the logistic classifier.
# max_iter is raised so the solver has plenty of room to settle on the scaled data.
sk_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# Fit the pipeline on the same predictors X and outcome y we gave statsmodels.
sk_model.fit(X, y)

# predict_proba returns a column for each class; column 1 is P(churn). Grab that column.
sk_prob = sk_model.predict_proba(X)[:, 1]

# Compare the two libraries' probabilities with a correlation: 1.0 would be perfect agreement.
agreement = np.corrcoef(sk_prob, churn_prob)[0, 1]
# Also report the average absolute gap between the two probability sets.
mean_gap = np.mean(np.abs(sk_prob - churn_prob.values))

# Print the comparison.
print(f"Correlation between sklearn and statsmodels probabilities: {agreement:.4f}")
print(f"Average absolute difference per customer                 : {mean_gap:.4f}")

**What we see.** The two probability sets agree almost perfectly — a correlation of about **0.998** and an average gap of about one percentage point per customer. That is the reassurance we wanted: `statsmodels` and `scikit-learn` are fitting the *same* logistic model; one simply specialises in the explanatory table, the other in the prediction-and-evaluation workflow.

**One model, one set of numbers.** From here on we will keep using **`churn_prob`** — the `statsmodels` probabilities — as our single reference, so that the coefficients and odds ratios we interpreted in §§5.10–5.11 and the evaluation numbers we are about to compute all come from one consistent model. With a probability in hand for every customer, the statistics are essentially done. Everything that remains is the genuinely hard part the manager warned us about: **turning a probability into a decision.**

***
## 5.14 From probability to decision — the threshold

The model says *"customer #1729 has a 0.62 probability of churning."* The retention team cannot phone "a 0.62". At some point a probability must become a **yes/no action**: call them, or do not. The rule that converts a probability into a decision is a **threshold** (also called a *cut-off*):

$$ \text{predict churn if } \; p_i \ge t, \qquad \text{predict stay if } \; p_i < t. $$

Reading it: $p_i$ is the customer's churn probability, $t$ is the threshold we choose, and $\ge$ means "greater than or equal to". The natural first choice is **$t = 0.5$** — *"flag anyone more likely than not to churn"* — and we will start there. But hold this thought, because it is the pivot of the whole back half of the notebook: **0.5 is a choice, not a law.** It happens to be the value that minimises raw mistakes when the two kinds of mistake are equally costly — and in churn they are emphatically *not* equally costly. We will return to move $t$ deliberately in §5.17. For now, let us apply $t = 0.5$ and see what decisions fall out.

In [ ]:
# Set our first threshold.
threshold = 0.5

# Turn each probability into a 0/1 decision: 1 (predict churn) if prob >= threshold, else 0.
predicted = (churn_prob >= threshold).astype(int)

# How many customers does this flag for a retention call?
n_flagged = int(predicted.sum())
# Print the count and what share of the customer base that is.
print(f"At threshold {threshold}, the model flags {n_flagged} customers "
      f"({n_flagged / n_total * 100:.1f}% of the base) for a retention call.")

At $t = 0.5$ the model flags **300 customers** — about **9.5%** of the base. (Already a hint of good news for our 10%-budget manager.) But a count alone hides *which* customers, and whether they are the right ones. To see that, we need the single most important table in classification: the **confusion matrix**.

***
## 5.15 The confusion matrix — the four ways a yes/no prediction can land

When both the truth and the prediction are yes/no, there are exactly **four** possible outcomes for each customer. Laying their counts in a 2×2 grid gives the **confusion matrix**, and every classification metric in the world is built from its four cells. The names are worth learning once, slowly, in churn terms:

- **True Positive (TP)** — we predicted *churn*, and they *did* churn. **A correct catch.** (We call; the call was warranted.)
- **True Negative (TN)** — we predicted *stay*, and they *did* stay. **A correct pass.** (We rightly leave them alone.)
- **False Positive (FP)** — we predicted *churn*, but they *stayed*. **A false alarm.** (We waste a call on someone who was fine — cost: the call.)
- **False Negative (FN)** — we predicted *stay*, but they *churned*. **A miss.** (We never called; they walked out the door — cost: a lost customer.)

The word *positive* always refers to **the thing we are trying to detect** — here, churn — not to anything good. A "false positive" is a false churn-alarm; a "false negative" is a churner we failed to flag. In churn the two errors are wildly unequal in cost: a false positive wastes a phone call, while a false negative loses a customer. Keep that asymmetry in mind — it is the reason §5.17 exists. `scikit-learn`'s `confusion_matrix` counts all four cells for us.

In [ ]:
# Compute the 2x2 confusion matrix. labels=[0,1] fixes the order: row/col 0 = stay, 1 = churn.
cm = confusion_matrix(y, predicted, labels=[0, 1])

# Unpack the four cells. .ravel() flattens [[TN, FP], [FN, TP]] into TN, FP, FN, TP in that order.
tn, fp, fn, tp = cm.ravel()

# Print the four counts in plain language.
print(f"True  Negatives (predicted stay,  stayed) : {tn}")
print(f"False Positives (predicted churn, stayed) : {fp}   <- wasted calls")
print(f"False Negatives (predicted stay,  churned): {fn}   <- missed churners")
print(f"True  Positives (predicted churn, churned): {tp}   <- correct catches")

# Draw the matrix as a labelled heatmap so the structure is visible at a glance.
plt.figure(figsize=(6, 5))
# annot=True writes the count in each cell; fmt="d" formats them as plain integers.
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["predicted: stay", "predicted: churn"],
            yticklabels=["actual: stay", "actual: churn"])
# Title the chart with the threshold so we remember which decision rule produced it.
plt.title(f"Confusion matrix at threshold t = {threshold}")
plt.show()

**What we see (at $t = 0.5$).** The four cells are:

|  | predicted: stay | predicted: churn |
|---|---|---|
| **actual: stay** | TN = 2585 | FP = 70 |
| **actual: churn** | FN = 265 | TP = 230 |

Read the bottom row, because it is where the money is: of the **495** customers who actually churned, the model **caught 230** (TP) and **missed 265** (FN). Read the top row for the cost side: of the customers who stayed, it correctly passed on **2,585** and raised a false alarm on only **70**. So the flag is *precise* (few false alarms) but lets *more than half the churners slip through*. Whether that is good or bad depends entirely on what we do with it — which brings us to the trap the manager warned us about.

***
## 5.16 Accuracy's trap, and the metrics that escape it

The most tempting single number is **accuracy**: the fraction of customers we got right.

$$ \text{accuracy} \;=\; \frac{TP + TN}{TP + TN + FP + FN}. $$

Reading it: the numerator is the two *correct* cells (correct catches plus correct passes); the denominator is all customers. Let us compute it — and then expose why, for a rare outcome, it lies.

In [ ]:
# Accuracy from the confusion matrix: correct cells over everything.
accuracy = (tp + tn) / (tp + tn + fp + fn)
# Print it.
print(f"Model accuracy at t = 0.5 : {accuracy:.4f}  ({accuracy*100:.1f}%)")

# The "lazy baseline": a model that predicts EVERYONE stays (never flags anyone).
# It is right on every stayer and wrong on every churner.
lazy_accuracy = n_stay / n_total
# Print it next to the model.
print(f"'Predict everyone stays'  : {lazy_accuracy:.4f}  ({lazy_accuracy*100:.1f}%)")
print()
print(f"The lazy model is {lazy_accuracy*100:.1f}% accurate while catching 0 of {n_churn} churners.")

**The trap, in one line.** A model that simply declares *"nobody ever churns"* is **84.3% accurate** on this data — because 84.3% of customers really do stay — while being **completely useless**: it catches **zero** churners and would let the retention team make **zero** useful calls. Our real model scores **89.4%**, only about five points higher. If accuracy were our scorecard, we would conclude the model is "a bit better than doing nothing" — a catastrophic misreading. **Accuracy is dominated by the majority class, so for a rare outcome it rewards a model for ignoring exactly the thing we care about.** This is precisely the trap the manager said had burned them before, and it is why the 15.7% base rate from §5.3 was worth memorising.

### The metrics that escape the trap

The fix is to stop collapsing the confusion matrix into one number and instead ask two *separate* questions — one about churners, one about stayers. Three metrics do the job:

$$ \text{sensitivity (recall)} = \frac{TP}{TP + FN}, \qquad \text{specificity} = \frac{TN}{TN + FP}, \qquad \text{precision} = \frac{TP}{TP + FP}. $$

In words, with the churn reading of each:

- **Sensitivity** (also called **recall** or the **true-positive rate**): *of all the customers who actually churned, what fraction did we catch?* It looks only at the bottom row of the matrix. This is the metric the retention team cares about most — a missed churner is a lost customer.
- **Specificity** (the **true-negative rate**): *of all the customers who actually stayed, what fraction did we correctly leave alone?* It looks only at the top row. High specificity means few wasted calls.
- **Precision**: *of all the customers we flagged for a call, what fraction really churned?* It looks at the predicted-churn *column*. Precision answers *"when we pick up the phone, how often is the call warranted?"*

Sensitivity and specificity pull against each other: flag more people and you catch more churners (sensitivity up) but also raise more false alarms (specificity down). That tension *is* the threshold decision. Let us compute all three at $t = 0.5$.

In [ ]:
# Sensitivity / recall: of all actual churners, the fraction we caught.
sensitivity = tp / (tp + fn)
# Specificity: of all actual stayers, the fraction we correctly left alone.
specificity = tn / (tn + fp)
# Precision: of all customers we flagged, the fraction who really churned.
precision = tp / (tp + fp)

# Print the three escape-the-trap metrics.
print(f"Sensitivity (recall) : {sensitivity:.4f}   <- caught {tp} of {tp+fn} churners")
print(f"Specificity          : {specificity:.4f}   <- correctly passed {tn} of {tn+fp} stayers")
print(f"Precision            : {precision:.4f}   <- of {tp+fp} flagged, {tp} really churned")

**What we see (at $t = 0.5$).** **Specificity is 0.974** — the model almost never bothers a happy customer. **Precision is 0.767** — when it does flag someone, it is right about three times out of four; those are calls worth making. But **sensitivity is only 0.465** — we are catching fewer than half of the customers who will actually leave. For a retention team whose whole job is to *stop* churn, missing 53% of churners is the number that should sting. The good news: sensitivity is not fixed. It is a direct consequence of where we put the threshold — and 0.5 was just a default. Time to move it on purpose.

***
## 5.17 Moving the threshold — the trade-off table

Lowering the threshold means flagging *more* customers (we act on weaker suspicions); raising it means flagging *fewer* (we act only on the near-certain). Every choice slides us along the same see-saw: **lower $t$ → higher sensitivity, lower specificity, lower precision; higher $t$ → the reverse.** There is no free lunch and no single "correct" threshold — only the one that best fits the *costs* of your situation. The honest way to choose is to lay the whole trade-off out in a table and look at it. Let us sweep $t$ from 0.1 to 0.9 and recompute the confusion-matrix metrics at each stop.

In [ ]:
# Define a helper that computes all the metrics at one threshold and returns them as a dict.
def metrics_at(t):
    # Decisions at this threshold.
    pred = (churn_prob >= t).astype(int)
    # Confusion-matrix cells.
    TN, FP, FN, TP = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    # Number flagged for a call = the predicted-churn column total.
    flagged = TP + FP
    # Return every number the manager might care about, rounded where helpful.
    return {
        "threshold": t,
        "flagged": flagged,
        "flagged_%": round(flagged / n_total * 100, 1),
        "caught (TP)": TP,
        "missed (FN)": FN,
        "false_alarms (FP)": FP,
        "sensitivity": round(TP / (TP + FN), 3),
        "specificity": round(TN / (TN + FP), 3),
        "precision": round(TP / (TP + FP), 3),
        "accuracy": round((TP + TN) / n_total, 3),
    }

# Sweep across a range of thresholds and stack the results into a table.
sweep = pd.DataFrame([metrics_at(t) for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]])
# Show the table without the DataFrame index, for a clean read.
print(sweep.to_string(index=False))

**What we see — read the table as a story.** Walk your eye down the rows:

- At **$t = 0.1$** we flag **1,074** customers (34% of the base) and catch **462 of 495** churners — a superb sensitivity of **0.93** — but precision collapses to **0.43**: well over half of our calls are wasted on people who were fine.
- At **$t = 0.5$** (our default) we flag only **300**, with precision **0.77** but sensitivity **0.46** — the high-precision, low-recall corner we already met.
- At **$t = 0.9$** we flag a mere **170** near-certain churners at **0.92** precision, but now we miss **338** of them.

The see-saw is right there in the columns: as the threshold climbs, **`caught` and `sensitivity` fall** while **`precision` and `specificity` rise**. Which row is "best"? It depends on the cost of a wasted call versus the cost of a lost customer — a business question, not a statistics question. Let us picture the trade-off before we let the budget decide.

In [ ]:
# A finer grid of thresholds for a smooth set of curves.
t_grid = np.linspace(0.02, 0.98, 97)
# Compute sensitivity, specificity and precision at each grid threshold.
sens_curve = [metrics_at(t)["sensitivity"] for t in t_grid]
spec_curve = [metrics_at(t)["specificity"] for t in t_grid]
prec_curve = [metrics_at(t)["precision"] for t in t_grid]

# Set up the figure.
plt.figure(figsize=(8, 5))
# Plot each metric against the threshold.
plt.plot(t_grid, sens_curve, label="sensitivity (recall)", linewidth=2)
plt.plot(t_grid, spec_curve, label="specificity", linewidth=2)
plt.plot(t_grid, prec_curve, label="precision", linewidth=2)
# Mark the default 0.5 threshold with a vertical line.
plt.axvline(0.5, color="gray", linestyle="--", label="t = 0.5 (default)")
# Label axes and title.
plt.xlabel("threshold  t")
plt.ylabel("metric value")
plt.title("The threshold trade-off: sensitivity falls as specificity and precision rise")
plt.legend()
plt.show()

**What we see.** The sensitivity curve slides **down** as we raise the threshold; specificity and precision climb **up**. They cross in the middle — there is no threshold that makes all three high at once. Choosing $t$ is choosing *where on these curves you want to live*, and that choice belongs to the business. But before we make it, there is a way to grade the model **independently of any single threshold** — by looking at *all* thresholds at once. That is the ROC curve.

***
## 5.18 The ROC curve and AUC — grading the model at every threshold at once

Every threshold gives one confusion matrix, and therefore one (sensitivity, specificity) pair — one row of the §5.17 table. The **ROC curve** (*Receiver Operating Characteristic*, a name inherited from World-War-II radar operators deciding whether a blip was a plane) plots *all* of those pairs as we sweep the threshold from high to low. By convention it plots:

- on the vertical axis, the **true-positive rate** = **sensitivity** (churners caught), and
- on the horizontal axis, the **false-positive rate** = **$1 - \text{specificity}$** (stayers falsely flagged).

Each point is one threshold. As we lower $t$ we move from the bottom-left (flag nobody: catch no churners, no false alarms) to the top-right (flag everybody: catch all churners, all false alarms). A **good** model bows hard toward the **top-left corner** — the dream spot where sensitivity is high *and* false alarms are low. A model with no skill produces the **diagonal line**, where catching churners costs an equal rate of false alarms — exactly what random guessing would give. Let us draw it.

In [ ]:
# roc_curve sweeps every threshold and returns the false-positive rate, the true-positive rate,
# and the thresholds that produced each point.
fpr, tpr, roc_thresholds = roc_curve(y, churn_prob)

# Set up the figure.
plt.figure(figsize=(7, 7))
# Plot the ROC curve itself.
plt.plot(fpr, tpr, color="darkorange", linewidth=2.5, label="logistic model")
# Plot the no-skill diagonal for reference.
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="random guessing (no skill)")
# Label the axes with both the technical and plain-English names.
plt.xlabel("false-positive rate  =  1 - specificity  (stayers falsely flagged)")
plt.ylabel("true-positive rate  =  sensitivity  (churners caught)")
plt.title("ROC curve: the model bows toward the top-left, far above random")
plt.legend(loc="lower right")
plt.show()

**What we see.** Our curve bows sharply toward the top-left and sits far above the diagonal — at a false-positive rate of only about 8% it is already catching roughly three-quarters of churners. That is a visibly strong classifier. To turn "how far above the diagonal" into a single number, we measure the **Area Under the Curve (AUC)**.

In [ ]:
# roc_auc_score computes the area under the ROC curve in one call.
auc = roc_auc_score(y, churn_prob)
# Print it.
print(f"AUC = {auc:.4f}")

**What AUC means, and what we got.** The AUC ranges from **0.5** (the diagonal — no skill, a coin flip) to **1.0** (a perfect classifier that bows all the way into the top-left corner). It has a beautifully concrete interpretation that needs no threshold at all:

> **AUC is the probability that the model gives a randomly chosen churner a higher churn score than a randomly chosen stayer.**

Our **AUC of about 0.935** therefore says: pick one real churner and one real stayer at random, and about **94% of the time** the model ranks the churner as the riskier of the two. As a rough industry yardstick, 0.5 is worthless, 0.7 is fair, 0.8 is good, and anything above 0.9 is excellent — so 0.935 is a genuinely strong result.

Two things make AUC the headline number for comparing classifiers. First, it is **threshold-free**: it grades the *ranking* of customers, before any cut-off is chosen, so it answers "is the model any good?" separately from "where should we set $t$?". Second, it is **robust to the rare-outcome trap** that wrecked accuracy in §5.16 — a model cannot inflate its AUC by ignoring the minority class. With the model's quality confirmed, we can finally make the manager's decision: pick the threshold that fits the budget.

***
## 5.19 The "call only 10%" policy — choosing a threshold to fit the budget

Now we honour the constraint from §5.1. The retention team can make about **315 calls** — 10% of 3,150 customers. So we do not pick the threshold by abstract preference; we pick the one that flags **as close to 315 customers as our budget allows**, and then we ask the only question that matters: *how many real churners does spending those 315 calls actually catch, and how much better is that than calling 315 people at random?*

Mechanically, "flag the 315 highest-risk customers" means: sort everyone by churn probability, and set the threshold at the probability of the 315th-riskiest customer. Let us find that threshold and grade the policy it implies.

In [ ]:
# The team's monthly calling budget: 10% of the customer base.
budget_frac = 0.10
budget_calls = int(round(budget_frac * n_total))

# The threshold that flags exactly the top `budget_calls` riskiest customers:
# sort probabilities high-to-low and read off the value at the budget-th position.
threshold_budget = np.sort(churn_prob.values)[::-1][budget_calls - 1]

# Apply that threshold to get the policy's decisions.
pred_budget = (churn_prob >= threshold_budget).astype(int)
# Confusion-matrix cells for the policy.
TN_b, FP_b, FN_b, TP_b = confusion_matrix(y, pred_budget, labels=[0, 1]).ravel()
# Number actually flagged (≈ budget).
flagged_b = TP_b + FP_b

# Policy metrics.
sens_b = TP_b / (TP_b + FN_b)             # share of all churners we reach
prec_b = TP_b / (TP_b + FP_b)             # share of our calls that were warranted

# Print the policy in business language.
print(f"Budget                       : {budget_calls} calls ({budget_frac*100:.0f}% of base)")
print(f"Threshold that fits the budget: p >= {threshold_budget:.3f}")
print(f"Customers flagged            : {flagged_b}")
print(f"Churners caught (TP)         : {TP_b} of {n_churn}  ({sens_b*100:.1f}% of all churners)")
print(f"Wasted calls (FP)            : {FP_b}")
print(f"Precision of the call list   : {prec_b:.3f}  ({prec_b*100:.1f}% of calls warranted)")

**What we see.** Spending the 315-call budget on the model's highest-risk customers reaches **237 of the 495 churners — about 48%** — at a precision of **75%** (three of every four calls land on a genuine churner). Now the punchline: how does that compare to the team's old habit of calling 315 customers at random?

In [ ]:
# If we called `budget_calls` people AT RANDOM, we'd expect to reach the base-rate fraction
# of churners — i.e. 15.7% of our calls would happen to be churners.
expected_random_catch = budget_frac * n_churn

# Lift = how many times more churners the model's list catches versus a random list of the same size.
lift = TP_b / expected_random_catch

# Print the comparison.
print(f"Random {budget_calls}-call list would catch about : {expected_random_catch:.0f} churners")
print(f"Model's {flagged_b}-call list actually catches      : {TP_b} churners")
print(f"Lift over random                              : {lift:.1f}x")

**What we see — the headline for the memo.** A random list of 315 calls would be expected to stumble onto only about **50 churners** (15.7% of 315). The model's list of the same size catches **237** — roughly a **4.8× improvement** for the *identical* amount of effort. That single sentence is the business case: the model does not let the team make *more* calls, it lets them make the **same** calls land on the right people nearly five times as often.

A closing word of honesty for the manager. At this budget we still **miss about half** of all churners (the 258 false negatives) — they simply fall below the top-315 cut-off. If churn is costly enough, the §5.17 table is the lever: stretching the budget to flag the top 20% (threshold ≈ 0.30) would catch about **388** churners (78%), at the price of more wasted calls. That is a resourcing decision for the manager, and our job is to put the trade-off in front of them honestly — not to hide it behind a single number. With the policy chosen and quantified, we are ready to let Claude help us *communicate* it.

***
## 5.20 Our API calls — Anthropic tool use for the churn-triage memo

> **Callback to §1.11, §2.13, §3.12, §4.16.** Session 01 made plain-text Anthropic calls. Session 02 asked Anthropic for a JSON outline and validated structure with Python-side checks. Session 03 stepped up to **schema-enforced** structured output, using **Anthropic's tool use** with a Pydantic schema (§3.12) — the model literally could not return something that did not match the schema. Session 04 reused that exact pattern for the drivers memo (§4.16). **Same pattern this week**, applied to a churn-triage action policy.

> **Why Anthropic only.** This course uses **Anthropic Claude exclusively** — there is no second provider, no second SDK, no second API key. Anthropic's Messages API plus tool use gives us everything we need: plain-text generation for language work, and schema-enforced output for structured deliverables. (This is course **Principle 20** in the instructor's notes.)

We will make **two API calls** this week, both to Anthropic:

1. **Call 1 — schema-enforced action policy.** Anthropic turns the evaluation numbers from §§5.15–5.19 into the structured triage policy the retention manager asked for, as a JSON object that *conforms to a Pydantic schema we define in Python*. The schema has four required fields: `recommended_policy`, `expected_impact`, `caveats`, and `what_would_strengthen`.
2. **Call 2 — plain-text headline paragraph.** Anthropic drafts the *"Headline"* paragraph of the stakeholder memo from the numbers we have already computed. *"Python computes, the model interprets"* — the model never invents numbers.

### Step 1 — verify the API key is available

In [ ]:
# Import os so we can read environment variables.
import os

# Read the Anthropic API key from the environment.
anth_key = os.environ.get("ANTHROPIC_API_KEY")

# If the key is missing, halt the notebook with a clear, actionable message.
if not anth_key:
    raise SystemExit(
        "ANTHROPIC_API_KEY is not set in your environment.\n"
        "Follow Step 4 of the repository README, close VS Code, reopen, and re-run this cell."
    )

# Confirm the key is set without printing its value.
print(f"ANTHROPIC_API_KEY is set. Key length: {len(anth_key)} characters.")

### Step 2 — create the Anthropic client

In [ ]:
# Import the official Anthropic Python SDK.
import anthropic

# Create the Anthropic client; it picks up ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

# Print a confirmation that the client object was created successfully.
print("Anthropic client ready.")

### Step 3 — define a Pydantic schema for the action policy

> **Callback to §3.12.** Pydantic gives us the JSON Schema for free. We write a regular Python class, and `.model_json_schema()` returns the JSON Schema dict Anthropic's tool-use feature expects. The same Pydantic class lets us re-wrap the validated response for type-safe access in the rest of the notebook.

The schema has four required string fields, matching the retention manager's brief:

- **`recommended_policy`** — who to call and at what probability cut-off, given the team's monthly budget.
- **`expected_impact`** — how many churners the policy reaches, at what precision, and the lift over random calling.
- **`caveats`** — what the manager must know before acting: how many churners we still miss, and the honest framing that the model **describes these customers** rather than forecasting future ones.
- **`what_would_strengthen`** — what additional data or follow-up would make the policy more trustworthy.

In [ ]:
# Import Pydantic's BaseModel and Field helpers (installed in the `ailab` venv as an Anthropic SDK dependency).
from pydantic import BaseModel, Field

# Define the structure of a churn-triage action policy as a Pydantic model.
class ChurnPolicy(BaseModel):
    recommended_policy: str = Field(
        description="The recommended calling policy in two to four sentences. State how many customers to call "
                    "(the team's monthly budget), at what probability cut-off, and which kinds of customers that "
                    "targets (reference the strongest drivers). Use only the numbers given to you."
    )
    expected_impact: str = Field(
        description="What the policy achieves, in two to four sentences. State how many churners it reaches, the "
                    "precision of the call list, and the lift over calling the same number of customers at random. "
                    "Use only the numbers given to you."
    )
    caveats: str = Field(
        description="The most important honesty caveats, two to four sentences. Cover: how many churners the policy "
                    "still misses; that the model DESCRIBES these customers and is not a forecast about future or "
                    "unseen customers; and that the drivers are associations, not proven causes. Do not invent numbers."
    )
    what_would_strengthen: str = Field(
        description="What additional data or follow-up would make the policy more trustworthy, two to four sentences. "
                    "Concrete and operational — e.g. recording the outcome of each call, gathering the reasons behind "
                    "complaints, or tracking the policy's results month over month."
    )

# Print confirmation.
print("Pydantic schema ChurnPolicy defined.")

### Step 4 — call Anthropic with a forced tool

The mechanics are exactly the same as §3.12 and §4.16. We build a tool spec from our Pydantic schema and tell Anthropic the model **must** call it (`tool_choice` forces the call). The response comes back as a list of content blocks; we scan for the `tool_use` block and pull its `input` — that dictionary is the validated action policy.

In [ ]:
# Safety re-fit — make the numbers this section needs available in scope, regardless of run order.
# Everything below was already computed earlier (the fit in §5.9, probabilities in §5.13, the
# confusion matrix in §5.15, the budget policy in §5.19), but re-deriving it here is cheap and makes
# this cell robust to running cells out of order or restarting the kernel mid-way through.
X = churn[predictors].astype(float)
y = churn["Churn"].astype(int)
X_const = sm.add_constant(X)
logit_res = sm.Logit(y, X_const).fit(disp=False, maxiter=200)
churn_prob = logit_res.predict(X_const)

# Odds ratios for the headline drivers.
odds_ratios = np.exp(logit_res.params)

# Evaluation at the default 0.5 threshold.
pred_05 = (churn_prob >= 0.5).astype(int)
tn5, fp5, fn5, tp5 = confusion_matrix(y, pred_05, labels=[0, 1]).ravel()
sens5 = tp5 / (tp5 + fn5)
prec5 = tp5 / (tp5 + fp5)

# The 10%-budget policy (315 calls).
budget_calls = int(round(0.10 * n_total))
threshold_budget = np.sort(churn_prob.values)[::-1][budget_calls - 1]
pred_b = (churn_prob >= threshold_budget).astype(int)
TNb, FPb, FNb, TPb = confusion_matrix(y, pred_b, labels=[0, 1]).ravel()
sensb = TPb / (TPb + FNb)
precb = TPb / (TPb + FPb)
liftb = TPb / (0.10 * n_churn)
auc_val = roc_auc_score(y, churn_prob)

# Build the numbers summary the model is allowed to see (Python computes, the model interprets).
policy_numbers = (
    f"Binomial logistic regression of customer churn, n = {n_total} customers "
    f"({n_churn} churned = {n_churn/n_total*100:.1f}% of the base; the rest stayed). "
    f"This is a description of THESE customers, not a forecast about other customers.\n\n"
    f"Model fit: McFadden pseudo-R^2 = {logit_res.prsquared:.3f}; ROC AUC = {auc_val:.3f} "
    f"(ranks a random churner above a random stayer about {auc_val*100:.0f}% of the time).\n\n"
    f"Strongest drivers, as odds ratios (>1 raises churn odds, <1 lowers them):\n"
    f"  Complains         OR = {odds_ratios['Complains']:.1f}   (a registered complaint multiplies churn odds enormously)\n"
    f"  Status            OR = {odds_ratios['Status']:.2f}    (non-active line status goes with much higher churn odds)\n"
    f"  Call_Failure      OR = {odds_ratios['Call_Failure']:.2f}    (each extra dropped call nudges churn odds up)\n"
    f"  Charge_Amount     OR = {odds_ratios['Charge_Amount']:.2f}    (a higher charge tier goes with LOWER churn odds)\n"
    f"  Frequency_of_use  OR = {odds_ratios['Frequency_of_use']:.3f}  (heavier callers churn slightly less)\n\n"
    f"Evaluation on all {n_total} customers:\n"
    f"  At threshold 0.5: flag {tp5 + fp5} customers, catch {tp5} of {n_churn} churners "
    f"({sens5*100:.0f}% sensitivity), precision {prec5*100:.0f}%.\n"
    f"  10% calling budget ({budget_calls} calls, threshold p >= {threshold_budget:.2f}): "
    f"catch {TPb} churners ({sensb*100:.0f}% of all churners) at {precb*100:.0f}% precision.\n"
    f"  That is about {liftb:.1f}x as many churners as calling {budget_calls} customers at random.\n"
    f"  Honesty: even at this budget the model MISSES {FNb} churners (they fall below the cut-off)."
)

# Build the Anthropic tool spec from our Pydantic schema.
# `model_json_schema()` returns a JSON Schema dict — exactly what Anthropic's `input_schema` field expects.
policy_tool = {
    "name": "submit_churn_policy",
    "description": "Submit a complete churn-triage action policy. You MUST call this tool with every required field filled in.",
    "input_schema": ChurnPolicy.model_json_schema(),
}

# Call Claude. `tool_choice` forces the model to call this specific tool — it cannot reply with plain text.
policy_response = client.messages.create(
    model="claude-haiku-4-5",                                          # same Haiku model used in §1.11, §2.13, §3.12, §4.16
    max_tokens=1600,                                                   # generous cap; the tool input is a few short paragraphs
    system=(                                                           # Retention Analyst persona
        "You are a Retention Analyst Assistant for a mobile telecom operator. "
        "You translate the computed results of a churn model into a structured, honest action policy "
        "for a retention team that can only call a limited number of customers each month. "
        "You enforce four disciplines: (1) recommend only actions supported by the numbers given to you; "
        "(2) describe THESE customers — never claim the model will generalise to future or unseen customers; "
        "(3) state plainly how many churners the policy still misses; "
        "(4) never invent numbers — use only those provided."
    ),
    tools=[policy_tool],                                               # the only tool the model is allowed to call
    tool_choice={"type": "tool", "name": "submit_churn_policy"},       # force this specific tool — no free-text reply
    messages=[
        {
            "role": "user",
            "content": (
                "Here are the results of a churn model fitted on a telecom customer base:\n\n"
                + policy_numbers
                + "\n\nPlease fill in the churn-triage action policy. Use the numbers above and only those."
            )
        }
    ],
)

# Scan the response for the tool_use block. Because tool_choice forced our tool, exactly one such block exists.
policy_dict = None
for block in policy_response.content:
    if block.type == "tool_use":
        policy_dict = block.input    # this dict already conforms to our schema — Anthropic enforced it
        break

# Wrap the dict back into our Pydantic class for type-safe access.
policy = ChurnPolicy(**policy_dict)

# Show the policy field by field for easy reading.
for field_name, value in policy.model_dump().items():
    print(f"-- {field_name} --")
    print(f"  {value}\n")

**What we see.** Anthropic returns a tool call whose `input` is a JSON object with exactly the four fields our Pydantic schema specified, each filled with a short, structured statement. The `tool_choice` guarantee means no validation step is needed afterwards — the model literally cannot return something that fails the schema. Same generation-time guarantee as §3.12's and §4.16's calls.

### Call 2 — Anthropic drafts the stakeholder memo paragraph

Now we hand the same computed numbers to Anthropic and ask for a short plain-text paragraph suitable for the *"Headline"* section of the one-page retention memo. Same *"Python computes, the model interprets"* rule.

In [ ]:
# Call Anthropic to write the stakeholder paragraph for the retention memo.
memo_response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    system=(
        "You translate churn-model results into one short paragraph for a telecom retention manager's memo. "
        "Lead with the headline policy: how many customers to call, how many churners that reaches, and the lift "
        "over random calling. Name the single strongest driver. Close with a one-sentence honesty caveat about how "
        "many churners the policy still misses, framed as a description of these customers rather than a forecast. "
        "Use plain English and plain percentages. Use ONLY the numbers given to you; never invent new numbers."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                "Here are the churn-model results:\n\n"
                + policy_numbers
                + "\n\nWrite ONE short paragraph (4 to 6 sentences) for the 'Headline' section of the "
                + "retention manager's memo. Lead with the calling policy and its lift over random, name the "
                + "strongest driver, and close with an honest caveat about the churners still missed."
            )
        }
    ]
)

# Print Anthropic's paragraph.
print(memo_response.content[0].text)

**What we see.** A short, plain-English paragraph that leads with the calling policy, names the lift over random, names the strongest driver, and closes with the honest *"how many we still miss"* caveat. The model **did not invent any numbers** — it was given the entire set it was allowed to use.

> **Mini-recap of §5.20.** Two API calls, one provider. Anthropic tool use with a Pydantic-derived schema **guarantees** the structured action policy at generation time. Plain-text Anthropic drafts the stakeholder paragraph from computed numbers. Same *"Python computes, the model interprets"* discipline as Sessions 01–04; same single-provider architecture as §3.12 and §4.16. (Principle 20.)

***
## 5.21 The action policy and stakeholder memo — fully worked

It is Friday afternoon. Here are the two artifacts you would hand the retention manager. Both are grounded entirely in numbers computed in this notebook.

---

### Artifact 1 — Churn-triage action policy (four-field structure)

**Recommended policy.** Each month, score all **3,150** customers with the fitted model and call the **315 highest-risk** of them — the team's 10% budget. In this customer base that means calling everyone above a churn probability of about **0.49**. The list is dominated by the model's strongest signals: customers who have **registered a complaint** (the single biggest driver — a complaint multiplies churn odds roughly **58-fold**), customers whose line **status** is non-active, and customers with more **call failures**. Higher-charge-tier customers are *less* likely to churn and naturally fall lower on the list.

**Expected impact.** That 315-call list reaches **237 of the 495 churners — about 48%** — at a precision of **75%** (three of every four calls land on a genuine churner). Calling **315 customers at random** would be expected to reach only about **50** churners (the 15.7% base rate), so the model delivers roughly a **4.8× lift** for the *identical* amount of effort. Across the whole base the model separates churners from stayers well: **AUC ≈ 0.94**, and it accounts for about half of the deviance (**McFadden pseudo-R² ≈ 0.49**).

**Caveats.** At this budget the policy still **misses 258 churners** — they simply fall below the top-315 cut-off. The model **describes these 3,150 customers**; it is not a forecast about future or unseen customers, and we make no claim that it will. The drivers are **associations, not proven causes**: a registered complaint is the loudest *marker* of imminent churn in this data, but the complaint and the churn may share a common upstream cause (a billing problem, a coverage gap) that the call alone will not fix.

**What would strengthen the policy.** First, **record the outcome of every call** — did the contacted customer stay? — so we can learn whether the intervention actually changes behaviour rather than just flags it. Second, **capture the reason behind each complaint**, since "complaint" is currently a single yes/no flag hiding very different situations. Third, **track the policy month over month**: if the share of calls that land on genuine churners drifts, that is the signal to re-examine the model with the manager.

---

### Artifact 2 — Stakeholder memo

**To:** Head of Retention
**From:** [Your name], Junior Analyst
**Re:** Churn-triage policy — who the team should call this month
**Date:** Friday, end of week 5

#### 1. What we analysed

A churn model fitted on **3,150** of our customers, of whom **495 (15.7%)** had already churned. The model reads eleven account signals — complaints, line status, call failures, charge tier, usage, tenure, customer value, and a few more — and returns, for each customer, a probability of churning.

#### 2. Headline

- With the team's **10% calling budget (315 calls)**, calling the model's highest-risk customers reaches **237 of our 495 churners — about 48% — at 75% precision** (three in four calls warranted).
- That is roughly **4.8× more churners** than calling 315 customers at random would reach (~50). Same effort, nearly five times the hit rate.
- The **single loudest signal is a registered complaint**: complainers churn at about **83%** versus **10%** for everyone else, and the model rates a complaint as multiplying churn odds roughly **58-fold**. Non-active line **status** and more **call failures** also raise risk; customers on **higher charge tiers** are *less* likely to churn.
- Overall the model separates churners from stayers well (**AUC ≈ 0.94**).

#### 3. Limitations

- **We still miss about half.** At the 315-call budget the policy leaves **258 churners** uncontacted — they fall below the cut-off. If churn is costly enough, stretching the budget to the top **20%** (≈ 630 calls, cut-off ≈ 0.30) would reach about **388 churners (78%)**, at the price of more wasted calls. That is a resourcing decision for you; our job is to put the trade-off in front of you honestly.
- **Description, not forecast.** These numbers describe **how well the model fits the customers we already have**. We make **no claim** about how it will behave on next month's customers — establishing that would be a separate piece of work.
- **Association, not causation.** A complaint is the loudest *marker* of churn, not proof that the complaint *causes* it. The call should diagnose the underlying problem, not just check a box.

#### 4. Recommendation

Adopt the **call-the-top-315** policy for one month as a trial. Prioritise complainers and non-active-status customers within that list — they carry the most risk. **Record the outcome of every call** so that next month we can measure whether contact actually retained the customer, not just whether we flagged them correctly.

#### 5. Open questions

- **Does contacting a flagged customer actually keep them?** The model finds who is at risk; it cannot tell us whether a call changes their mind. A small trial with recorded outcomes would answer this.
- **What is hiding inside the "complaint" flag?** It is currently a single yes/no. Capturing the reason would let us route calls to the right fix.
- **Which lever moves churn the most?** Complaints, call failures, and charge tier are all associated with churn; only a deliberate intervention can tell us which one, if changed, actually reduces it.

---

That is your two-artifact deliverable. **Save your own memo as `_reports/session05_churn_memo.md`** in your course directory. The instructor will review it next week.

> **Mini-recap of §5.21.** Two artifacts: a structured action policy (generated with Anthropic tool use, grounded in the evaluation of §§5.15–5.19) and a stakeholder memo (drafted with Anthropic on numbers we computed ourselves). Every claim traces back to a cell in this notebook. The structure prevents the most common triage-memo mistakes — overstating the model as a crystal ball, hiding the missed churners below the fold, and confusing a loud marker with a proven cause.

***
## 5.22 References — what to study to deepen this session

A focused list this week. **Pick three or four** of the StatQuest videos. The single most useful trio is **Odds and Log(Odds)**, **Logistic Regression**, and **Maximum Likelihood** — once those three land, every formula in this notebook becomes a re-reading rather than a first encounter.

### StatQuest videos (YouTube)

| Video | What it clarifies |
|---|---|
| [Odds and Log(Odds), Clearly Explained](https://www.youtube.com/watch?v=ARfXDSkQf1Y) | The bridge from probability to the linear scale a regression can live on. Pairs with §5.5. |
| [Odds Ratios and Log(Odds Ratios), Clearly Explained](https://www.youtube.com/watch?v=8nm0G-1uJzA) | Exactly how to read $e^\beta$ as "multiply the odds by this". Pairs with §5.11. |
| [Logistic Regression, Clearly Explained](https://www.youtube.com/watch?v=yIYKR4sgzI8) | The whole model end-to-end — the sigmoid, the linear predictor, the fit. Pairs with §§5.6–5.9. |
| [Maximum Likelihood, Clearly Explained](https://www.youtube.com/watch?v=XepXtl9YKwc) | Why we maximise likelihood instead of minimising squared error. The twin of least squares from §5.8. |
| [Maximum Likelihood for the Binomial Distribution](https://www.youtube.com/watch?v=4KKV9yZCoM4) | MLE worked on coin flips — the Bernoulli/Binomial bridge back to Session 01. Pairs with §§5.7–5.8. |
| [The Confusion Matrix, Clearly Explained](https://www.youtube.com/watch?v=Kdsp6soqA7o) | TP, FP, FN, TN in pictures. Pairs with §5.15. |
| [Sensitivity and Specificity, Clearly Explained](https://www.youtube.com/watch?v=vP06aMoz4v8) | The two error rates that the accuracy number hides. Pairs with §5.16. |
| [ROC and AUC, Clearly Explained](https://www.youtube.com/watch?v=4jRBRDbJemM) | Why we sweep the threshold and summarise with one area. Pairs with §§5.17–5.18. |

### Documentation

- [statsmodels `Logit`](https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.Logit.html) — the interpretable summary table we read in §§5.9–5.12 (coefficients, z-tests, pseudo-R²).
- [scikit-learn `LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) — `predict_proba` and the pipeline we used in §5.13.
- [scikit-learn `confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html), [`roc_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html), and [`roc_auc_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html) — the evaluation tools of §§5.15–5.18.
- [Anthropic tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use) — the schema-enforced output mechanism behind §5.20's action policy.

### Dataset

- [Iranian Churn dataset, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/563/iranian+churn+dataset) — 3,150 customers of an Iranian telecom operator, licensed **CC BY 4.0**. The `Churn` column and eleven account features are exactly the ones we modelled.

> A good week: **Odds and Log(Odds)** plus **Logistic Regression** to lock in §§5.5–5.9, **Maximum Likelihood** for the §5.8 intuition, and **ROC and AUC** for §§5.17–5.18. That covers the spine of every classification model you will meet later.

See you in **Session 06**, where we stay with classification but let the model draw **curved** decision boundaries — the straight-line-on-the-log-odds-scale assumption of this week is the next thing we will relax. The confusion matrix, threshold, and ROC/AUC machinery you built today carries straight through.

<hr>

![](../_img/DK_Logo_White_150.png)

DataKolektiv, 2026.

[hello@datakolektiv.com](mailto:hello@datakolektiv.com)

<font size=1>License: [GPLv3](../LICENSE). This Notebook is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version. This Notebook is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.</font>